# **Statistical Analysis & Modelling**

## Objectives

* Apply statistical tests to analyse relationships between Spotify audio features and song popularity
* Build a simple machine learning model to predict song popularity

## Inputs

* Clean Spotify dataset produced from Notebook 1
* Selected variables including popularity, danceability, energy, acousticness, and tempo

## Outputs

* Results of hypothesis tests (Spearman correlation, Welch's t-test, ANOVA)
* Visualisations supporting the statistical analysis
* A machine learning model predicting song popularity
* Model evaluation results

## Additional Comments

* This notebook focuses on statistical analysis and modelling using the cleaned dataset created in Notebook 1.

---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [ ]:
import os
current_dir = os.getcwd()
current_dir

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [ ]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

Confirm the new current directory

In [ ]:
current_dir = os.getcwd()
current_dir

# Section 4 - Hypothesis Tests

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
import pingouin as pg

In [13]:
df = pd.read_csv("/Users/priyanatt1/Documents/VSC-projects/spotify-popularity-analysis/Data/clean/spotify_clean.csv") 
df.head()

,popularity,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,tempo,valence
0,73,0.676,0.4610,-6.746,0.1430,0.0322,0.000001,0.3580,87.917,0.715
1,55,0.420,0.1660,-17.235,0.0763,0.9240,0.000006,0.1010,77.489,0.267
2,57,0.438,0.3590,-9.734,0.0557,0.2100,0.000000,0.1170,76.332,0.120
3,71,0.266,0.0596,-18.515,0.0363,0.9050,0.000071,0.1320,181.740,0.143
4,82,0.618,0.4430,-9.681,0.0526,0.4690,0.000000,0.0829,119.949,0.167


---

### Hypothesis 1 – Energy and Song Popularity

H₀: Average song popularity is equal across low, medium, and high energy songs.

H₁: At least one energy group has a different average song popularity.


Test: One-way ANOVA

In the context of the dataset: Songs will be grouped into low, medium, and high energy categories, and the test will be used to evaluate whether average popularity differs across these groups.

In [3]:
df['energy_group'] = pd.cut(
    df['energy'],
    bins=[0, 0.33, 0.66, 1],
    labels=['Low Energy', 'Medium Energy', 'High Energy']
)

df['energy_group'].value_counts()
# Create energy groups

energy_group
High Energy      60885
Medium Energy    37279
Low Energy       15835
Name: count, dtype: int64

In [4]:
pg.anova(data=df, dv='popularity', between='energy_group', detailed=True)

,Source,SS,DF,MS,F,p_unc,np2
0,energy_group,3.273222e+05,2,163661.078837,330.856871,5.322521e-144,0.005771
1,Within,5.638906e+07,113996,494.658245,NaN,NaN,NaN


### What this code does

- The code groups songs into three categories based on their energy levels: low, medium, and high energy using `pd.cut()`.
- A one-way ANOVA test is used to compare the average popularity across these three energy groups.
- The test calculates a F-statistic and a p-value.
- The p-value shows whether the differences in average popularity between the energy groups are statistically significant.


### Interpretation

The one-way ANOVA test produced a p-value of 5.32e-144, which is less than the significance level of 0.05 therefore the null hypothesis is rejected and the alternative hypothesis is accapted.

In the context of the dataset:
- This result suggests that song energy levels are associated with differences in song popularity. Songs with different energy levels tend to have different average popularity scores, indicating that energy may play a role in how popular a track becomes.

---

### Hypothesis 2 – Acousticness and Song Popularity

H₀: Songs with high acousticness and low acousticness have equal average popularity.

H₁: Songs with high acousticness have different average popularity compared to songs with low acousticness.

Test: Mann-Whitney U Test.


In the context of the dataset: songs will be grouped into low and high acousticness categories, and the test will evaluate whether the popularity scores differ between these groups.

In [10]:
# Create acousticness groups
df['acousticness_group'] = pd.cut(
    df['acousticness'],
    bins=[0, 0.5, 1],
    labels=['Low Acousticness', 'High Acousticness']
)

df['acousticness_group'].value_counts()

acousticness_group
Low Acousticness     80144
High Acousticness    33817
Name: count, dtype: int64

In [12]:
pg.mwu(
    x=df[df['acousticness_group'] == 'Low Acousticness']['popularity'],
    y=df[df['acousticness_group'] == 'High Acousticness']['popularity']
)

,U_val,alternative,p_val,RBC,CLES
MWU,1.404450e+09,two-sided,2.060026e-22,0.036407,0.518203


### What this code does

- The code groups songs into two categories based on acousticness: low acousticness and high acousticness using `pd.cut()`.
- A Mann-Whitney U test is used to compare the popularity scores between these two groups.
- The test calculates a U statistic and a p-value.
- The p-value indicates whether the difference in popularity between the two acousticness groups is statistically significant.

### Interpretation

The Mann-Whitney U test produced a p-value of 2.06e-22, which less than the significance level of 0.05 therefore the null hypothesis is rejected and the alternative hypothesis is accepted.

In the context of the dataset: 
- Songs with high acousticness and low acousticness have significantly different popularity scores, suggesting that acousticness may influence how popular a song becomes.

---

### Hypothesis 3 – Tempo and Popular Songs

H₀: Tempo range and song popularity category are independent.

H₁: Tempo range and song popularity category are associated.

Test: Chi-Squared Test.

In the context of the dataset:songs will be grouped into tempo ranges (slow, medium, and fast) and popularity will be categorised into popular and not popular songs. The test will evaluate whether the distribution of popular songs differs across tempo ranges.

In [18]:
df['tempo_group'] = pd.cut(
    df['tempo'],
    bins=[0, 90, 130, 250],
    labels=['Slow', 'Medium', 'Fast']
)

df['tempo_group'].value_counts()

tempo_group
Medium    55976
Fast      41622
Slow      16245
Name: count, dtype: int64

In [19]:
df['popularity_group'] = np.where(
    df['popularity'] >= 70,
    'Popular',
    'Not Popular'
)

df['popularity_group'].value_counts()

popularity_group
Not Popular    108528
Popular          5472
Name: count, dtype: int64

In [20]:
expected, observed, stats = pg.chi2_independence(
    data=df,
    x='tempo_group',
    y='popularity_group'
)

stats

,test,lambda,chi2,dof,pval,cramer,power
0,pearson,1.000000,179.832381,2.0,8.910347e-40,0.039717,1.0
1,cressie-read,0.666667,179.983565,2.0,8.261626e-40,0.039734,1.0
2,log-likelihood,0.000000,180.597585,2.0,6.077610e-40,0.039802,1.0
3,freeman-tukey,-0.500000,181.334751,2.0,4.203966e-40,0.039883,1.0
4,mod-log-likelihood,-1.000000,182.313752,2.0,2.576748e-40,0.039991,1.0
5,neyman,-2.000000,185.018893,2.0,6.662817e-41,0.040286,1.0


In [21]:
stats.query("test == 'pearson'")['pval']

0    8.910347e-40
Name: pval, dtype: float64

### What This Code Does

- The code groups songs into three tempo categories (slow, medium, and fast) using `pd.cut()`.
- Song popularity is also grouped into two categories: popular and not popular using `np.where()`.
- A Chi-Squared test is then carried out using `pg.chi2_independence()`.
- The `stats` table shows the Chi-Squared test results.
- The Pearson p-value is then extracted using `stats.query("test == 'pearson'")['pval']`.
- The p-value shows whether tempo group and popularity category are statistically associated.

### Interpretation

The Chi-Squared test produced a p-value of 8.91e-40, which is less than the significance level of 0.05 therefore the null hypothesis is rejected and the alternative hypothesis is accepted.

In the context of the dataset:

- Tempo range and song popularity category are statistically associated.
- This suggests that certain tempo ranges may be more common among popular songs.

---

# Section 5 - Machine Learning Model

### Problem definition

Predict song popularity using Spotify audio features including danceability, energy, loudness, speechiness, acousticness, instrumentalness, liveness, tempo and valence.

### Why this is appropriate

Popularity is a continuous variable ranging from 0–100, making regression the appropriate modelling approach.

<u>In the context of the dataset:</u> the goal is to determine whether Spotify audio features can be used to predict how popular a song is likely to be.

### Model choice

A Linear Regression model is used as a baseline predictive model. Linear regression is simple, interpretable and suitable for examining relationships between audio features and popularity scores.

---

# Section 6 - Conclusions, Reflections & Limitations

---

# Push files to Repo

* In cases where you don't need to push files to Repo, you may replace this section with "Conclusions and Next Steps" and state your conclusions and next steps.

In [ ]:
import os
try:
  # create your folder here
  # os.makedirs(name='')
except Exception as e:
  print(e)
